# KArSL: download raw videos and extract MediaPipe landmarks (free Colab)

Runs on a free Colab CPU or GPU runtime. The landmark extractor is CPU-bound, so a GPU is not required.
Output: one small `.npz` per clip under `landmarks/<signID>/<signer>_<clip>.npz`, zipped and copied to Google Drive.

Steps: clone repo -> install -> download KArSL raw videos (public Drive folder) -> `scripts/extract_landmarks.py` -> zip to Drive.

In [ ]:
!git clone -q https://github.com/meshalbr/saudi-sign-language-recognition.git
%cd saudi-sign-language-recognition
!pip install -q -e . gdown
!python scripts/download_models.py

In [ ]:
# KArSL public folders (https://hamzah-luqman.github.io/KArSL/). Start with KArSL-100 (signs 0071-0170).
KARSL_RAW_VIDEOS = "https://drive.google.com/drive/folders/1R8ZI6a1xgRNqUThDFA-X4RT7FJh9WeRr"
SIGNER = "01"   # download one signer at a time to stay inside Colab disk limits

import gdown, os
os.makedirs("data/raw", exist_ok=True)
# gdown can list and fetch a public folder; large folders may need --remaining-ok and reruns.
gdown.download_folder(KARSL_RAW_VIDEOS, output="data/raw", quiet=False, remaining_ok=True)
!du -sh data/raw && find data/raw -type f | head -5

KArSL file names encode `<signID>_<signer>_<repetition>`; the extractor expects `<videos>/<label>/<signer>_<x>.mp4`.
The next cell re-arranges whatever structure arrived into that layout (adjust the regex once you see the real names).

In [ ]:
import re, shutil
from pathlib import Path

src = Path("data/raw"); dst = Path("data/videos")
pat = re.compile(r"(?P<sign>\d{4})[_-](?P<signer>\d{2})[_-](?P<rep>\d{2,4})", re.I)
n = 0
for f in src.rglob("*"):
    if f.suffix.lower() not in {".mp4", ".avi", ".mov", ".mkv"}:
        continue
    m = pat.search(f.stem)
    if not m:
        print("unparsed name:", f.name); continue
    out = dst / m["sign"] / f"s{m['signer']}_{m['rep']}{f.suffix.lower()}"
    out.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(f), out); n += 1
print("arranged", n, "videos into", dst)

In [ ]:
!python scripts/extract_landmarks.py --videos data/videos --out landmarks --signer-from-name --max-frames 200

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/ssl && zip -qr /content/drive/MyDrive/ssl/karsl_landmarks.zip landmarks
!ls -lh /content/drive/MyDrive/ssl/

On the laptop: download `karsl_landmarks.zip` from Drive, unzip into `data/landmarks/`, then
`python -m ssl_recognition.train --config configs/baseline.yaml --test-signer s03`.